In [15]:
import os 
from langchain_community.vectorstores import FAISS
from google import genai
from dotenv import load_dotenv
from my_embeddings import BentechEmbeddings

In [16]:

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
custom_embeddings = BentechEmbeddings(api_key=api_key)

vector_db=FAISS.load_local(
    "faiss_bentech_index",
    custom_embeddings,
    allow_dangerous_deserialization=True
)
retriever = vector_db.as_retriever(search_kwargs={"k":3})


In [17]:
def ask_bentech_ai(question):
    # a. Kullanıcının sorusuyla alakalı dökümanları FAISS'ten çek
    related_docs = retriever.invoke(question)
    
    # b. Çekilen bu parçaların metin içeriklerini birleştir (Context oluştur)
    context_text = "\n\n".join([doc.page_content for doc in related_docs])
    
    # c. Gemini'yi yönlendirecek "Sistem ve Bağlam" Promptunu hazırla
    prompt = f"""
    Sen Bentech AI & Craft şirketinin kurumsal asistanısın. 
    Aşağıdaki "BAĞLAM" başlığı altındaki bilgileri kullanarak kullanıcının sorusunu yanıtla. 
    Sorunun cevabı bu bağlamda yoksa, kafandan uydurma ve "Bu bilgi kurumsal el kitabında yer almıyor" de.
    
    BAĞLAM:
    {context_text}
    
    SORU: 
    {question}
    
    CEVAP:
    """
    
    # d. Yeni Google SDK ile Gemini'den yanıt üret (Generation)
    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model="gemini-2.0-flash", # Hızlı ve akıllı güncel model
        contents=prompt
    )
    
    return response.text

# --- CANLI TEST ALANI ---
soru = "Yeni başlayan bir yazılım mühendisiyim, bana hangi donanımlar verilecek ve hangi günler ofiste olmalıyım?"
cevap = ask_bentech_ai(soru)

print("=" * 40)
print(f"SORU: {soru}")
print(f"CEVAP:\n{cevap}")
print("=" * 40)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}